In [4]:
pip install torch_geometric


  Using cached requests-2.32.5-py3-none-any.whl.metadata (4.9 kB)
  Using cached attrs-25.4.0-py3-none-any.whl.metadata (10 kB)
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 17.9 MB/s  0:00:00
Using cached attrs-25.4.0-py3-none-any.whl (67 kB)
Using cached idna-3.11-py3-none-any.whl (71 kB)
Using cached requests-2.32.5-py3-none-any.whl (64 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15/15 [torch_geometric] [torch_geometric]er]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install ldpc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 52.1 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.9/196.9 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 626.1/626.1 kB 36.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 93.4 MB/s eta 0:00:00:00:01


In [2]:
pip install torch

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 MB 34.6 MB/s  0:00:02m0:00:0100:01
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [torch]32m4/5 [torch]]ols]

[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
import numpy as np
import torch
from torch_geometric.data import Data

def load_parity_check_txt_to_pyg(filename: str) -> Data:
    """
    Reads your file format:
      first line: m n
      next m lines: variable indices for each check row
    Returns a PyG Data with a bipartite Tanner graph:
      - nodes: (n variable nodes) + (m check nodes)
      - edges: undirected (we store as two directed edges for message passing)
    """
    with open(filename, "r") as f:
        first = f.readline().strip().split()
        m, n = int(first[0]), int(first[1])

        # edges between check r and variable c
        var_nodes = []
        chk_nodes = []
        for r in range(m):
            line = f.readline()
            cols = [int(x) for x in line.strip().split()]
            for c in cols:
                var_nodes.append(c)        # variable node index [0..n-1]
                chk_nodes.append(n + r)    # check node index [n..n+m-1]

    # Build edge_index (2, E*2) as directed edges both ways
    src = torch.tensor(var_nodes + chk_nodes, dtype=torch.long)
    dst = torch.tensor(chk_nodes + var_nodes, dtype=torch.long)
    edge_index = torch.stack([src, dst], dim=0)

    num_nodes = n + m

    # Basic node features:
    # node_type: variable=[1,0], check=[0,1]
    x = torch.zeros((num_nodes, 2), dtype=torch.float)
    x[:n, 0] = 1.0
    x[n:, 1] = 1.0

    # Edge features placeholder (we will augment with action markings later)
    # We'll store 1 edge feature: is_action_edge (0/1), initialize 0
    edge_attr = torch.zeros((edge_index.size(1), 1), dtype=torch.float)

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr)
    data.n_vars = n
    data.n_checks = m
    return data

/Users/manyyeon/data/git/optimizing-qLDPC-code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
def mark_edge_swap_action(data: Data, v1: int, c1_row: int, v2: int, c2_row: int) -> Data:
    """
    Marks the two selected edges and their endpoints in node/edge features.

    v1, v2 are variable indices in [0, n_vars-1]
    c1_row, c2_row are check ROW indices in [0, n_checks-1]
      (we convert them to node ids n_vars + row)

    This modifies:
      - node features: adds is_action_endpoint (1 dim)
      - edge features: sets is_action_edge on the two edges (both directions)
    """
    n = data.n_vars
    m = data.n_checks
    c1 = n + c1_row
    c2 = n + c2_row

    # Add/overwrite endpoint marker feature
    # x: [node_type(2)] -> [node_type(2), is_action_endpoint(1)]
    if data.x.size(1) == 2:
        endpoint = torch.zeros((data.num_nodes, 1), dtype=torch.float, device=data.x.device)
        data.x = torch.cat([data.x, endpoint], dim=1)

    data.x[:, 2] = 0.0
    data.x[v1, 2] = 1.0
    data.x[v2, 2] = 1.0
    data.x[c1, 2] = 1.0
    data.x[c2, 2] = 1.0

    # Reset edge_attr to all zeros then mark selected edges
    data.edge_attr[:, 0] = 0.0

    # Find the indices in edge_index corresponding to (v -> c) and (c -> v)
    ei = data.edge_index
    # mask for v1 <-> c1 and v2 <-> c2, both directions
    mask1 = ((ei[0] == v1) & (ei[1] == c1)) | ((ei[0] == c1) & (ei[1] == v1))
    mask2 = ((ei[0] == v2) & (ei[1] == c2)) | ((ei[0] == c2) & (ei[1] == v2))
    data.edge_attr[mask1, 0] = 1.0
    data.edge_attr[mask2, 0] = 1.0

    return data

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GINEConv, global_add_pool

class GNNQNetwork(nn.Module):
    def __init__(self, node_in_dim: int, edge_in_dim: int, hidden_dim: int = 64, num_layers: int = 6):
        super().__init__()
        self.node_encoder = nn.Sequential(
            nn.Linear(node_in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.edge_encoder = nn.Sequential(
            nn.Linear(edge_in_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.convs = nn.ModuleList()
        for _ in range(num_layers):
            mlp = nn.Sequential(
                nn.Linear(hidden_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, hidden_dim),
            )
            self.convs.append(GINEConv(nn=mlp))

        self.readout = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)  # scalar Q
        )

    def forward(self, data):
        # data.x: [num_nodes, node_in_dim]
        # data.edge_attr: [num_edges, edge_in_dim]
        # node features and edge features are encoded to embeddings separately before message passing
        x = self.node_encoder(data.x)
        e = self.edge_encoder(data.edge_attr)

        # Message passing layers
        for conv in self.convs:
            x_new = conv(x, data.edge_index, e)
            x = F.relu(x + x_new)  # residual connection

        # If you're using a single graph at a time, batch is all zeros
        batch = getattr(data, "batch", torch.zeros(data.num_nodes, dtype=torch.long, device=x.device))
        g = global_add_pool(x, batch)   # [batch_size, hidden_dim]

        q = self.readout(g)             # [batch_size, 1]
        return q.squeeze(-1)            # [batch_size]

In [8]:
import torch
import torch.nn.functional as F
from torch_geometric.data import Batch

def dqn_loss(
    q_net,
    target_net,
    data_s_a,                  # Data for (s,a)
    reward,                    # scalar tensor
    data_sprime_candidates,    # list[Data] for (s', a') candidates
    done=False,                # bool
    gamma=0.99
):
    # Q(s,a)
    q_sa = q_net(data_s_a)  # shape: [1] because GNNQNetwork output is q.squeeze(-1)

    with torch.no_grad():
        if done or len(data_sprime_candidates) == 0:
            q_next = torch.tensor(0.0, device=q_sa.device) # shape: []
        else:
            batch = Batch.from_data_list(data_sprime_candidates)
            q_all = target_net(batch)          # shape: [num_candidates]
            q_next = q_all.max() # shape: [] if num_candidates=1, or more generally, a scalar representing the max Q-value

        # Ensure reward is a tensor and unsqueeze to [1] if it's a scalar []
        if not isinstance(reward, torch.Tensor):
            reward = torch.tensor(reward, dtype=torch.float, device=q_sa.device)

        # Ensure y has shape [1] for consistency with q_sa
        y = (reward + gamma * q_next).unsqueeze(0) # Now y is [1]

    # Huber loss is usually more stable for value function regression than MSE
    return F.smooth_l1_loss(q_sa, y)

In [9]:
import random
import copy
import torch
from torch_geometric.data import Data, Batch

def get_var_to_check_edges(data: Data):
    """Return list of (v, cnode) edges where v in [0..n-1], cnode in [n..n+m-1]."""
    n = data.n_vars
    src, dst = data.edge_index
    mask = (src < n) & (dst >= n)   # variable -> check
    v = src[mask].tolist()
    c = dst[mask].tolist()
    return list(zip(v, c))

def edge_set_var_to_check(data: Data):
    """Set of edges (v, cnode) for fast membership checks."""
    return set(get_var_to_check_edges(data))

In [10]:
def apply_edge_swap(data: Data, v1: int, c1: int, v2: int, c2: int) -> Data:
    n = data.n_vars
    assert v1 < n and v2 < n and c1 >= n and c2 >= n

    if v1 == v2 or c1 == c2:
        return None

    Eset = edge_set_var_to_check(data)  # should store (v, c) with c>=n
    if (v1, c1) not in Eset or (v2, c2) not in Eset:
        return None
    if (v1, c2) in Eset or (v2, c1) in Eset:
        return None

    new_data = copy.deepcopy(data)
    src = new_data.edge_index[0].clone()
    dst = new_data.edge_index[1].clone()

    def replace_directed(u_old, v_old, u_new, v_new):
        mask = (src == u_old) & (dst == v_old)
        src[mask] = u_new
        dst[mask] = v_new

    # Replace the 4 directed edges corresponding to the two undirected edges:
    # (v1<->c1) becomes (v1<->c2)
    replace_directed(v1, c1, v1, c2)   # v1 -> c1  => v1 -> c2
    replace_directed(c1, v1, c2, v1)   # c1 -> v1  => c2 -> v1

    # (v2<->c2) becomes (v2<->c1)
    replace_directed(v2, c2, v2, c1)
    replace_directed(c2, v2, c1, v2)

    new_data.edge_index = torch.stack([src, dst], dim=0)
    return new_data

In [12]:
def sample_random_valid_swap(data: Data, max_tries: int = 200):
    edges = get_var_to_check_edges(data)
    Eset = set(edges)
    n = data.n_vars

    for _ in range(max_tries):
        (v1, c1) = random.choice(edges)
        (v2, c2) = random.choice(edges)
        if v1 == v2 or c1 == c2:
            continue
        # reject if would create duplicates
        if (v1, c2) in Eset or (v2, c1) in Eset:
            continue
        # ok
        return (v1, c1, v2, c2)
    return None

In [13]:
def mark_action_by_nodes(data: Data, v1, c1_node, v2, c2_node) -> Data:
    n = data.n_vars
    c1_row = c1_node - n
    c2_row = c2_node - n
    return mark_edge_swap_action(data, v1=v1, c1_row=c1_row, v2=v2, c2_row=c2_row)

In [14]:
from collections import deque

class ReplayBuffer:
    def __init__(self, capacity=50000):
        self.buf = deque(maxlen=capacity)

    def add(self, s, a, r, sprime, done):
        self.buf.append((s, a, r, sprime, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        return batch

    def __len__(self):
        return len(self.buf)

In [ ]:
from optimization.analyze_codes.decoder_performance_from_state import evaluate_performance_of_state
import math
import time
import torch.optim as optim
import torch
import ldpc
import ldpc.code_util
from scipy.sparse import csr_matrix
import numpy as np # Added for np.uint8 and np.ones
import random # Added for random.randint

def compute_distance_stub(data: Data) -> float:
    # Extract n and m from the PyG Data object
    n = data.n_vars
    m = data.n_checks

    # Filter edge_index to get only variable-to-check edges
    # Variable nodes are 0 to n-1, Check nodes are n to n+m-1
    src, dst = data.edge_index
    var_to_check_mask = (src < n) & (dst >= n)

    var_indices = src[var_to_check_mask].cpu().numpy()
    check_node_indices = dst[var_to_check_mask].cpu().numpy()

    # Convert check_node_indices to check_row_indices (0 to m-1)
    check_row_indices = check_node_indices - n

    # Create the H matrix in CSR format
    # The data for csr_matrix are the ones (presence of edge)
    # The indices are the column indices (variable indices)
    # The indptr specifies where each row starts in the indices and data arrays
    # For direct construction with rows, cols, data, use coo_matrix first
    H = csr_matrix(
        (np.ones_like(var_indices, dtype=np.uint8),
         (check_row_indices, var_indices)),
        shape=(m, n),
        dtype=np.uint8
    )

    # Classical Parameters
    n_cl, k_cl, _ = ldpc.code_util.compute_code_parameters(H)
    d_cl = ldpc.code_util.compute_exact_code_distance(H)
    r_cl = ldpc.mod2.rank(H)

    # Transpose Parameters
    if k_cl == n_cl - H.shape[0]:
        n_T_cl = H.shape[0]
        k_T_cl = n_T_cl - r_cl
        d_T_cl = np.inf
    else:
        n_T_cl, k_T_cl, d_T_cl = ldpc.code_util.compute_code_parameters(
            csr_matrix(H.T, dtype=np.uint8))

    n_q = n_cl**2 + n_T_cl**2
    k_q = k_cl**2 + k_T_cl**2
    d_q = min(d_cl, d_T_cl)  # Lower bound estimate
    return d_q

def ler_mc_estimate(data: Data, MC_budget=10000, p=0.03):
    H = data_to_H_csr(data)
    cost_result = evaluate_performance_of_state(state=H, p_vals=[p], MC_budget=MC_budget, run_label='', canskip=False)
    ler = cost_result['logical_error_rates'][0]
    return ler

def generate_next_candidates(sprime: Data, K: int = 64):
    """
    Generate K candidate next states from s' by applying random valid swaps.
    """
    cands = []
    for _ in range(K):
        a = sample_random_valid_swap(sprime)
        if a is None:
            continue
        v1, c1, v2, c2 = a
        d = copy.deepcopy(sprime)
        d = mark_action_by_nodes(d, v1, c1, v2, c2)
        cands.append(d)
    return cands

# target_net ← q_net
# This is a common operation in DQN where the target network is periodically updated to match the current Q-network.
def hard_update(target_net, q_net):
    target_net.load_state_dict(q_net.state_dict())

def data_to_H_csr(data: Data):
    n = data.n_vars
    m = data.n_checks
    src, dst = data.edge_index
    mask = (src < n) & (dst >= n)
    var = src[mask].cpu().numpy()
    chk = (dst[mask].cpu().numpy() - n)
    return csr_matrix((np.ones_like(var, dtype=np.uint8), (chk, var)), shape=(m, n), dtype=np.uint8)

def train_dqn(
    initial_data: Data,
    q_net,
    target_net,
    steps=2000,
    episode_len=50,
    batch_size=16,
    gamma=0.99,
    lr=1e-3,
    start_learning=200,
    target_update_every=200,
    K_act=49,
    K_next=49,
    epsilon=1.0,
    epsilon_final=0,
    epsilon_decay=0.999,
    p=0.03,
    MC_budget=5000,
    A = 1.0, # reward scaling factor for distance change
    beta = 0.1 # reward scaling factor for LER change
):
    start_time = time.time()
    device = next(q_net.parameters()).device
    print(f"Training on device: {device}")
    q_net.train()
    target_net.eval()

    opt = optim.Adam(q_net.parameters(), lr=lr)
    replay = ReplayBuffer()

    s = copy.deepcopy(initial_data)
    d_old = compute_distance_stub(s)
    p_old = ler_mc_estimate(data=s, MC_budget=MC_budget, p=p)

    t_in_ep = 0

    best_d = d_old
    best_p_big = float("inf")
    best_state = None

    for step in range(steps):
        # ----- choose action (epsilon-greedy) -----
        if random.random() < epsilon:
            print(f"Step {step}: Choosing random action (epsilon={epsilon:.3f})")
            a = sample_random_valid_swap(s)
        else:
            print(f"Step {step}: Choosing action with Q-network (epsilon={epsilon:.3f})")
            cands = []
            acts = []
            for _ in range(K_act):
                cand_a = sample_random_valid_swap(s)
                if cand_a is None:
                    continue
                v1,c1,v2,c2 = cand_a
                d_marked = mark_action_by_nodes(copy.deepcopy(s), v1, c1, v2, c2)
                cands.append(d_marked)
                acts.append(cand_a)

            if len(cands) == 0:
                a = None
            else:
                batch = Batch.from_data_list([d.to(device) for d in cands])
                with torch.no_grad():
                    qvals = q_net(batch)
                a = acts[int(qvals.argmax().item())]

        if a is None:
            # If no valid action found, reset episode
            s = copy.deepcopy(initial_data)
            d_old = compute_distance_stub(s)
            p_old = ler_mc_estimate(data=s, MC_budget=MC_budget, p=p)
            t_in_ep = 0
            continue

        v1, c1, v2, c2 = a

        # ----- environment step -----
        sprime = apply_edge_swap(s, v1, c1, v2, c2)
        if sprime is None:
            # invalid transition, treat as no-op with small penalty
            r = -0.1
            done = False
            sprime = s
            d_new = d_old
            p_new = p_old
        else:
            d_new = compute_distance_stub(sprime)

            delta_d = float(d_new - d_old)

            # LER shaping only when distance doesn't change
            eps = 1e-12

            if delta_d > 0:
                print(f"Step {step}: Distance increased from {d_old} to {d_new}.")
                r = A * delta_d
                p_new = p_old

                if d_new > best_d:
                    best_d = d_new
                    r += delta_d # extra reward for new best distance
                    print(f"New best distance at step {step}: {best_d}")
            elif delta_d < 0:
                print(f"Step {step}: Distance decreased from {d_old} to {d_new}.")
                r = A * delta_d
                p_new = p_old
            else:
                print(f"Step {step}: Distance unchanged at {d_old}.")
                p_new = ler_mc_estimate(data=sprime, MC_budget=MC_budget, p=p)

                r_ler = math.log(p_old + eps) - math.log(p_new + eps)  # improvement positive
                r = beta * r_ler
            done = (t_in_ep + 1 >= episode_len)

        # store transition (store raw graphs + action tuple)
        replay.add(copy.deepcopy(s), (v1, c1, v2, c2), r, copy.deepcopy(sprime), done)

        # move forward
        s = sprime
        d_old = d_new
        p_old = p_new
        t_in_ep = (t_in_ep + 1) % episode_len

        # ----- learning step -----
        if len(replay) >= start_learning:
            print(f"Step {step}: Starting learning with replay buffer size {len(replay)}")
            batch = replay.sample(batch_size)

            losses = []
            opt.zero_grad()

            for (sb, ab, rb, sprimeb, doneb) in batch:
                v1b, c1b, v2b, c2b = ab

                data_s_a = mark_action_by_nodes(copy.deepcopy(sb), v1b, c1b, v2b, c2b).to(device)

                # next-state candidates to approximate max_{a'} Q(s',a')
                sprime_cands = generate_next_candidates(sprimeb, K=K_next)
                sprime_cands = [d.to(device) for d in sprime_cands]

                loss = dqn_loss(
                    q_net=q_net,
                    target_net=target_net,
                    data_s_a=data_s_a,
                    reward=torch.tensor(rb, dtype=torch.float, device=device),
                    data_sprime_candidates=sprime_cands,
                    done=doneb,
                    gamma=gamma
                )
                losses.append(loss)

            total_loss = torch.stack(losses).mean()
            total_loss.backward()
            opt.step()

        # ----- target update -----
        if step % target_update_every == 0 and step > 0:
            hard_update(target_net, q_net)

        # ----- epsilon schedule -----
        epsilon = max(epsilon_final, epsilon * epsilon_decay)

    end_time = time.time()
    total_time = end_time - start_time
    print(f"Training completed in {total_time // 3600}h {total_time % 3600 // 60}m {total_time % 60}s.")
    return q_net

# Run Example

In [17]:
from optimization.experiments_settings import codes, path_to_initial_codes, textfiles

C = 2
initial_data = load_parity_check_txt_to_pyg(path_to_initial_codes + textfiles[C])

# Move initial_data to the appropriate device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
initial_data = initial_data.to(device)

print(f"Loaded initial graph with {initial_data.num_nodes} nodes and {initial_data.num_edges} edges.")
print(f"Number of variables: {initial_data.n_vars}, Number of checks: {initial_data.n_checks}")

Loaded initial graph with 56 nodes and 192 edges.
Number of variables: 32, Number of checks: 24


In [19]:
# Initialize Q-network and Target Network
# node_in_dim should be 3 if 'is_action_endpoint' feature is added, otherwise 2.
# Let's assume the mark_edge_swap_action function will handle adding the feature if it's not present.
# So we'll first mark an action to ensure the correct dimension.

# Create a dummy marked data to get correct dimensions
dummy_data = mark_edge_swap_action(copy.deepcopy(initial_data), v1=0, c1_row=0, v2=1, c2_row=1)

node_in_dim = dummy_data.x.size(1)
edge_in_dim = dummy_data.edge_attr.size(1)
hidden_dim = 64
num_layers = 6

q_net = GNNQNetwork(node_in_dim, edge_in_dim, hidden_dim, num_layers).to(device)
target_net = GNNQNetwork(node_in_dim, edge_in_dim, hidden_dim, num_layers).to(device)

# Perform initial hard update
hard_update(target_net, q_net)

print(f"Q-network initialized with node_in_dim={node_in_dim}, edge_in_dim={edge_in_dim}")
print("Target network initialized and synchronized.")

Q-network initialized with node_in_dim=3, edge_in_dim=1
Target network initialized and synchronized.


In [22]:
# Train the DQN agent
print("Starting DQN training...")

trained_q_net = train_dqn(
    initial_data=initial_data,
    q_net=q_net,
    target_net=target_net,
    steps=1000, # You can adjust the number of training steps
    episode_len=50,
    batch_size=32,
    gamma=0.99, # discount factor for future rewards
    lr=1e-4,
    start_learning=50, # Start learning after populating replay buffer
    target_update_every=50,
    K_act=49,
    K_next=49,
    epsilon=1.0,
    epsilon_final=0,
    epsilon_decay=0.99,
    p=0.03,
    MC_budget=1000,
    A = 1.0, # reward scaling factor for distance change
    beta = 0.1 # reward scaling factor for LER change
)

print("DQN training finished.")

Starting DQN training...
Training on device: cpu
H: [32, 8, 6]
H^T: [24, 0, inf]
[[1600, 64, 6]]
{'n_classical': 32, 'k_classical': 8, 'd_classical': 6, 'n_T_classical': 24, 'k_T_classical': 0, 'd_T_classical': inf, 'rank_H': 24, 'n_quantum': 1600, 'k_quantum': 64, 'd_quantum': 6}


/Users/manyyeon/data/git/optimizing-qLDPC-code/.venv/lib/python3.12/site-packages/ldpc/code_util/code_util.py:164: UserWarning: This function has exponential complexity. Not recommended for large pcms. Use the                            'ldpc.code_util.estimate_code_distance' function instead.
  warnings.warn(


Hx, Hz, Lx, Lz: (768, 1600), (768, 1600), (64, 1600), (64, 1600)
BP max iterations: 160, OSD order: 2, MS scaling factor: 0.625
Decoder  finished in 0.0m 2.64s with 11 failures out of 1000 runs.
Logical error rate for : 0.011 ± 0.0033000 (stderr)
Step 0: Choosing random action (epsilon=1.000)
Step 0: Distance unchanged at 6.
H: [32, 8, 6]
H^T: [24, 0, inf]
[[1600, 64, 6]]
{'n_classical': 32, 'k_classical': 8, 'd_classical': 6, 'n_T_classical': 24, 'k_T_classical': 0, 'd_T_classical': inf, 'rank_H': 24, 'n_quantum': 1600, 'k_quantum': 64, 'd_quantum': 6}
Hx, Hz, Lx, Lz: (768, 1600), (768, 1600), (64, 1600), (64, 1600)
BP max iterations: 160, OSD order: 2, MS scaling factor: 0.625
Decoder  finished in 0.0m 2.11s with 13 failures out of 1000 runs.
Logical error rate for : 0.013 ± 0.0035838 (stderr)
Step 1: Choosing random action (epsilon=0.990)
Step 1: Distance unchanged at 6.
H: [32, 8, 6]
H^T: [24, 0, inf]
[[1600, 64, 6]]
{'n_classical': 32, 'k_classical': 8, 'd_classical': 6, 'n_T_clas

# Save the trained Q Network

In [27]:
import torch
torch.save(trained_q_net.state_dict(), "qnet_gnn_dqn.pt")

In [29]:
import torch

# Assuming you have the q_net initialized with the correct architecture (as before)
# node_in_dim, edge_in_dim, hidden_dim, num_layers should be the same as when saved.
# If you run into issues, you might need to re-run the cell where q_net is initialized.

# Create a dummy data for dimension inference if the previous cell defining q_net is not run.
dummy_data_for_init = mark_edge_swap_action(copy.deepcopy(initial_data), v1=0, c1_row=0, v2=1, c2_row=1)
node_in_dim_loaded = dummy_data_for_init.x.size(1)
edge_in_dim_loaded = dummy_data_for_init.edge_attr.size(1)
hidden_dim_loaded = 64 # Match the hidden_dim used during training
num_layers_loaded = 6  # Match the num_layers used during training

loaded_q_net = GNNQNetwork(node_in_dim_loaded, edge_in_dim_loaded, hidden_dim_loaded, num_layers_loaded).to(device)
loaded_q_net.load_state_dict(torch.load("qnet_gnn_dqn.pt"))
loaded_q_net.eval() # Set to evaluation mode if you're not planning further training

print("Model loaded successfully!")


Model loaded successfully!


In [30]:
from scipy.sparse import csr_matrix
import numpy as np

def pyg_data_to_csr_H(data):
    n = data.n_vars
    m = data.n_checks
    src, dst = data.edge_index

    # only var -> check directed edges
    mask = (src < n) & (dst >= n)
    var = src[mask].cpu().numpy()
    chk = (dst[mask].cpu().numpy() - n)  # row indices

    H = csr_matrix(
        (np.ones_like(var, dtype=np.uint8), (chk, var)),
        shape=(m, n),
        dtype=np.uint8
    )
    return H

In [31]:
def save_H_as_txt(H_csr, out_path):
    m, n = H_csr.shape
    H_csr = H_csr.tocsr()
    with open(out_path, "w") as f:
        f.write(f"{m} {n}\n")
        for r in range(m):
            cols = H_csr.indices[H_csr.indptr[r]:H_csr.indptr[r+1]]
            f.write(" ".join(map(str, cols.tolist())) + "\n")

In [32]:
import copy, torch

@torch.no_grad()
def rollout_and_keep_best(initial_data, q_net, steps=200, K_act=49, device="cpu"):
    q_net.eval()

    s = copy.deepcopy(initial_data)
    d = compute_distance_stub(s)
    best_ler = ler_mc_estimate(s, MC_budget=10000, p=0.03)
    best_d = d
    best_state = copy.deepcopy(s)

    for t in range(steps):
        # choose best action among K_act samples
        best_a, best_q = None, -1e9
        for _ in range(K_act):
            a = sample_random_valid_swap(s)
            if a is None:
                continue
            v1, c1, v2, c2 = a
            s_a = mark_action_by_nodes(copy.deepcopy(s), v1, c1, v2, c2).to(device)
            q = float(q_net(s_a).item())
            if q > best_q:
                best_q, best_a = q, a

        if best_a is None:
            break

        v1, c1, v2, c2 = best_a
        sprime = apply_edge_swap(s, v1, c1, v2, c2)
        if sprime is None:
            continue

        s = sprime
        d = compute_distance_stub(s)
        print(f"Step {t+1}: Distance = {d} (best {best_d})")

        if d >= best_d:
            best_d = d
            ler = ler_mc_estimate(s, MC_budget=10000, p=0.03)
            print(f"Step {t+1}: LER = {ler:.2e} (best {best_ler:.2e})")
            if best_ler > ler:
                best_ler = ler
                best_state = copy.deepcopy(s)
        

    return best_state, best_d, best_ler

In [34]:
best_overall_d = -1
best_overall_state = None

for i in range(1):
    st, bd, bl = rollout_and_keep_best(initial_data, q_net, steps=1000, K_act=49, device=device)
    print(f"rollout {i}: best d = {bd}, best_ler = {bl}")
    if bd > best_overall_d:
        best_overall_d = bd
        best_overall_state = st

print("BEST overall d found:", best_overall_d)

H: [32, 8, 6]
H^T: [24, 0, inf]
[[1600, 64, 6]]
{'n_classical': 32, 'k_classical': 8, 'd_classical': 6, 'n_T_classical': 24, 'k_T_classical': 0, 'd_T_classical': inf, 'rank_H': 24, 'n_quantum': 1600, 'k_quantum': 64, 'd_quantum': 6}
Hx, Hz, Lx, Lz: (768, 1600), (768, 1600), (64, 1600), (64, 1600)
BP max iterations: 160, OSD order: 2, MS scaling factor: 0.625
Decoder  finished in 0.0m 20.27s with 126 failures out of 10000 runs.
Logical error rate for : 0.0126 ± 0.0011155 (stderr)
Step 1: Distance = 6 (best 6)
H: [32, 8, 6]
H^T: [24, 0, inf]
[[1600, 64, 6]]
{'n_classical': 32, 'k_classical': 8, 'd_classical': 6, 'n_T_classical': 24, 'k_T_classical': 0, 'd_T_classical': inf, 'rank_H': 24, 'n_quantum': 1600, 'k_quantum': 64, 'd_quantum': 6}
Hx, Hz, Lx, Lz: (768, 1600), (768, 1600), (64, 1600), (64, 1600)
BP max iterations: 160, OSD order: 2, MS scaling factor: 0.625
Decoder  finished in 0.0m 19.98s with 116 failures out of 10000 runs.
Logical error rate for : 0.0116 ± 0.0010708 (stderr)
St

In [35]:
H_best = pyg_data_to_csr_H(best_overall_state)

# (optional) compute “real” distance here
d_cl = ldpc.code_util.compute_exact_code_distance(H_best)  # for small only
# d_T  = ldpc.code_util.compute_exact_code_distance(H_best.T.tocsr())
# d_T = math.inf
# print("Exact d_cl, d_T:", d_cl, d_T, "min:", min(d_cl, d_T))
print("Exact d_cl:", d_cl)

save_H_as_txt(H_best, "best_candidate_from_rl.txt")

Exact d_cl: 10


In [36]:
import h5py
from optimization.analyze_codes.decoder_performance_from_state import evaluate_performance_of_state
from optimization.experiments_settings import from_edgelist, load_tanner_graph

code_name = "[1600,64]"  # Update this key if needed
MC_budget = int(1e5)
filepath = "optimization/results/best_candidate_from_rl_C2.txt"
p = 0.03

state = load_tanner_graph(filepath)

cost_result = evaluate_performance_of_state(state=state, p_vals=[p], MC_budget=MC_budget, run_label='', canskip=False)

H: [32, 8, 10]
H^T: [24, 0, inf]
[[1600, 64, 10]]
{'n_classical': 32, 'k_classical': 8, 'd_classical': 10, 'n_T_classical': 24, 'k_T_classical': 0, 'd_T_classical': inf, 'rank_H': 24, 'n_quantum': 1600, 'k_quantum': 64, 'd_quantum': 10}
Hx, Hz, Lx, Lz: (768, 1600), (768, 1600), (64, 1600), (64, 1600)
BP max iterations: 160, OSD order: 2, MS scaling factor: 0.625
Decoder  finished in 3.0m 59.89s with 200 failures out of 100000 runs.
Logical error rate for : 0.002 ± 0.0001413 (stderr)
